# 03 — Hypothesis Testing

Three core hypotheses (with two supporting tests) are pre-specified, tested with
non-parametric methods, corrected for multiple comparisons, and reported with
effect sizes so that statistical and practical significance are separated.

| # | Hypothesis | Test |
|---|---|---|
| H1 | Advertised price differs across room types within a city | Kruskal–Wallis + Dunn |
| H2 | Capacity / bedrooms are associated with price within a city | Spearman ρ |
| H3 | Listing ratings differ across room types / property families | Kruskal–Wallis |
| H4 | Review activity differs across room types | Kruskal–Wallis |
| H5 | Host response rate differs across cities | Kruskal–Wallis |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from src.config import ANALYSIS_DATASET, CITY_ORDER
from src.statistics import hypothesis_suite

df = pd.read_parquet(ANALYSIS_DATASET)
df["city"] = pd.Categorical(df["city"].astype(str), categories=CITY_ORDER, ordered=True)
print(f"Rows: {len(df):,} | price-analysable: {int(df['price_analysable'].sum()):,}")

Rows: 279,599 | price-analysable: 276,846


## H1 — Price by room type (within city, local currency)

**H₀:** the price distribution is identical across room types.
**H₁:** at least one room type differs.
Kruskal–Wallis is used because prices are heavily right-skewed; Dunn's test
provides pairwise comparisons with Bonferroni control; ε² is the effect size.

In [2]:
res = hypothesis_suite(df, CITY_ORDER)
h1 = res["h1_kruskal_price_roomtype"].sort_values("epsilon_squared", ascending=False)
h1[["city", "currency", "n", "H", "p_value", "p_bh", "epsilon_squared", "effect"]]

,city,currency,n,H,p_value,p_bh,epsilon_squared,effect
2,Sydney,AUD,33293,14460.342915,0.000000e+00,0.000000e+00,0.434298,large
1,New York,USD,36619,15357.199318,0.000000e+00,0.000000e+00,0.419342,large
6,Mexico City,MXN,19857,6665.270080,0.000000e+00,0.000000e+00,0.335580,large
9,Hong Kong,HKD,7017,1931.215782,0.000000e+00,0.000000e+00,0.274949,large
5,Istanbul,TRY,24271,5328.182818,0.000000e+00,0.000000e+00,0.219441,large
3,Rome,EUR,27367,4272.980221,0.000000e+00,0.000000e+00,0.156049,large
4,Rio de Janeiro,BRL,26341,4054.524765,0.000000e+00,0.000000e+00,0.153834,large
8,Cape Town,ZAR,18895,2641.066240,0.000000e+00,0.000000e+00,0.139647,medium
0,Paris,EUR,64020,5321.530662,0.000000e+00,0.000000e+00,0.083081,medium
7,Bangkok,THB,19166,968.131530,1.473210e-209,1.473210e-209,0.050367,small


In [3]:
dunn = res["h1_dunn_price_roomtype"]
print(f"Pairwise comparisons: {len(dunn):,} | significant after Bonferroni: {int(dunn['significant_5pct'].sum()):,}")
dunn[dunn["significant_5pct"]].sort_values("p_bonferroni").head(15)

Pairwise comparisons: 60 | significant after Bonferroni: 57


,city,currency,group_1,group_2,n_1,n_2,median_1,median_2,mean_rank_1,mean_rank_2,z,p_value,p_bonferroni,significant_5pct
0,Paris,EUR,Entire place,Hotel room,54925,1252,81.0,156.0,33390.582622,50294.680511,-32.011955,0.0,0.0,True
50,Cape Town,ZAR,Hotel room,Private room,297,4494,1300.0,620.5,10739.621212,5984.310636,14.551790,0.0,0.0,True
25,Rio de Janeiro,BRL,Entire place,Shared room,19051,606,346.0,100.0,15001.431185,6004.793729,28.674986,0.0,0.0,True
30,Istanbul,TRY,Entire place,Private room,12267,10537,349.0,161.0,15194.897815,8800.496583,68.712058,0.0,0.0,True
31,Istanbul,TRY,Entire place,Shared room,12267,680,349.0,100.0,15194.897815,6013.691912,33.262103,0.0,0.0,True
32,Istanbul,TRY,Hotel room,Private room,787,10537,319.0,161.0,14405.211563,8800.496583,21.647631,0.0,0.0,True
33,Istanbul,TRY,Hotel room,Shared room,787,680,319.0,100.0,14405.211563,6013.691912,22.875906,0.0,0.0,True
34,Istanbul,TRY,Private room,Shared room,10537,680,161.0,100.0,8800.496583,6013.691912,10.052916,0.0,0.0,True
55,Hong Kong,HKD,Entire place,Shared room,2560,426,680.0,180.5,4837.068164,1550.328638,31.007687,0.0,0.0,True
36,Mexico City,MXN,Entire place,Private room,10412,8835,950.0,400.0,12994.621590,6434.610979,79.119756,0.0,0.0,True


**Interpretation.** If H1 is rejected in most cities, room type is a reliable
price differentiator, and the post-hoc pairs identify which categories separate.
With tens of thousands of listings, p-values will be tiny; ε² is the guard
against overstating a trivial difference.

## H2 — Capacity and bedrooms vs price (within city)

In [4]:
res["h2_spearman_capacity_price"], res["h2_spearman_bedrooms_price"]

(             city  variable      n       rho  p_value    ci_low   ci_high  p_bh
 0           Paris  capacity  64020  0.526028      0.0  0.520063  0.531942   0.0
 1        New York  capacity  36619  0.551896      0.0  0.544300  0.559401   0.0
 2          Sydney  capacity  33293  0.679459      0.0  0.673282  0.685542   0.0
 3            Rome  capacity  27367  0.494474      0.0  0.484926  0.503903   0.0
 4  Rio de Janeiro  capacity  26341  0.511883      0.0  0.502374  0.521269   0.0
 5        Istanbul  capacity  24271  0.505939      0.0  0.495950  0.515795   0.0
 6     Mexico City  capacity  19857  0.556709      0.0  0.546451  0.566801   0.0
 7         Bangkok  capacity  19166  0.307843      0.0  0.294195  0.321366   0.0
 8       Cape Town  capacity  18895  0.664448      0.0  0.655920  0.672805   0.0
 9       Hong Kong  capacity   7017  0.514825      0.0  0.496361  0.532824   0.0,
              city  variable      n       rho  p_value    ci_low   ci_high  p_bh
 0           Paris  bedroom

In [5]:
pd.concat([
    res["h2_spearman_capacity_price"].assign(variable="accommodates"),
    res["h2_spearman_bedrooms_price"].assign(variable="bedrooms"),
]).pivot(index="city", columns="variable", values="rho").round(3)

variable,accommodates,bedrooms
city,,
Bangkok,0.308,0.424
Cape Town,0.664,0.661
Hong Kong,0.515,0.488
Istanbul,0.506,0.367
Mexico City,0.557,0.462
New York,0.552,0.448
Paris,0.526,0.478
Rio de Janeiro,0.512,0.563
Rome,0.494,0.433


**Interpretation.** ρ measures monotonic association, not a causal effect. A
moderate ρ means capacity explains part of the price signal; the rest comes from
location, property type, amenities and host positioning.

## H3 — Ratings by room type and property family

In [6]:
res["h3_kruskal_rating_roomtype"].sort_values("city")

,city,n,k,H,p_value,epsilon_squared,effect,p_bh
7,Bangkok,11181,4,22.170118,6.012466e-05,0.001715,negligible,6.680518e-05
8,Cape Town,13435,4,14.992873,1.822750e-03,0.000893,negligible,1.822750e-03
9,Hong Kong,3778,4,88.081758,5.656158e-19,0.022544,small,1.131232e-18
5,Istanbul,11229,4,161.574691,8.380919e-35,0.014127,small,4.190459e-34
6,Mexico City,14484,4,91.974229,8.250312e-20,0.006145,negligible,2.062578e-19
1,New York,26777,4,173.089499,2.739403e-37,0.006353,negligible,2.739403e-36
0,Paris,48036,4,73.563625,7.360735e-16,0.001469,negligible,1.226789e-15
4,Rio de Janeiro,16118,4,152.883355,6.291219e-33,0.009301,negligible,2.097073e-32
3,Rome,20893,4,66.636857,2.239568e-14,0.003046,negligible,3.199383e-14
2,Sydney,22376,4,65.158845,4.638541e-14,0.002778,negligible,5.798177e-14


In [7]:
res["h3b_kruskal_rating_property"].sort_values("city")

,city,n,k,H,p_value,epsilon_squared,effect,p_bh
7,Bangkok,11154,11,159.266572,4.591524e-29,0.013396,small,1.147881e-28
8,Cape Town,13407,15,334.111782,8.793721e-63,0.023903,small,8.793721e-62
9,Hong Kong,3720,9,37.634381,8.794047e-06,0.007986,negligible,8.794047e-06
5,Istanbul,11182,10,249.288369,1.409918e-48,0.021508,small,4.699726e-48
6,Mexico City,14423,9,81.996573,1.936260e-14,0.005134,negligible,2.766085e-14
1,New York,26716,10,293.065218,7.713440e-58,0.010637,small,3.856720e-57
0,Paris,48010,10,109.431157,1.919654e-19,0.002092,negligible,3.199423e-19
4,Rio de Janeiro,16064,10,63.192279,3.237788e-10,0.003376,negligible,4.047235e-10
3,Rome,20853,11,50.707780,1.977175e-07,0.001953,negligible,2.196861e-07
2,Sydney,22319,13,148.921179,9.383812e-26,0.006138,negligible,1.876762e-25


**Interpretation.** Ratings are compressed near the top of the scale, so even a
statistically significant difference can be practically small. Effect sizes here
are expected to be small; that is itself a finding, not a failed test.

## H4 — Review activity by room type

In [8]:
res["h4_kruskal_reviews_roomtype"].sort_values("city")

,city,n,k,H,p_value,epsilon_squared,effect,p_bh
7,Bangkok,19360,4,1120.525926,1.282109e-242,0.057735,small,2.564217e-242
8,Cape Town,19086,4,1282.777179,8.032092e-278,0.067067,medium,2.008023e-277
9,Hong Kong,7087,4,265.672122,2.665449e-57,0.037085,small,3.331811e-57
5,Istanbul,24517,4,1856.549332,0.000000e+00,0.075615,medium,0.000000e+00
6,Mexico City,20058,4,1807.166934,0.000000e+00,0.089965,medium,0.000000e+00
1,New York,36984,4,91.758806,9.178047e-20,0.002400,negligible,9.178047e-20
0,Paris,64628,4,199.343367,5.848457e-43,0.003038,negligible,6.498286e-43
4,Rio de Janeiro,26608,4,443.902747,6.828137e-96,0.016573,small,9.754482e-96
3,Rome,27641,4,1641.894969,0.000000e+00,0.059301,small,0.000000e+00
2,Sydney,33630,4,1033.877142,8.054251e-224,0.030657,small,1.342375e-223


**Interpretation.** Review count is a proxy for booking-related activity, not
satisfaction. It is confounded by listing age and market size.

## H5 — Host response rate across cities

In [9]:
res["h5_kruskal_host_response_city"]

,n,k,H,p_value,epsilon_squared,effect
0,150898,10,1970.205989,0.0,0.012998,small


**Interpretation.** A significant result describes differences in reported host
behaviour across markets. It is not a service-quality measure.

## Method notes

- Benjamini–Hochberg controls the false-discovery rate across the repeated
  per-city tests.
- Kruskal–Wallis tests stochastic dominance and assumes broadly similar
  distribution shapes; it is not a test of equal medians.
- Listings are treated as independent. Multi-listing hosts violate this mildly,
  so p-values are optimistic.
- Excluding prices above the 99th percentile prevents a handful of data errors
  from dominating rank-based tests.
- Association is not causation.